In [ ]:
# HORIZON environment probe (docs/STATE.md, amendment s).  Runs once, apart from the campaign: it reads
# the Kaggle image's digest and Python, installs kaggle/requirements-runner.txt from the code tag with the
# runner's own command, and records the exact version of every package the runner's requirements need.
# It runs no search and holds no credential.  It leaves probe.json and console.log in the working folder.
# Since 14 November 2026 (code tag automation-v4) it also checks the kind gvgaisearch's environment: JPype imports,
# the pinned Temurin JDK is fetched and checked by its sha256, GVGAI and the bridge compile, and the synthetic level
# corridor 0 of the tests loads.  TAG is the ref the probe installs from; scripts/automation/envprobe.py push --ref
# writes it into the copy it pushes.
import json, os, re, shutil, subprocess, sys, platform, time
from importlib import metadata
TAG = "automation-v2"
REPO_URL = "https://github.com/rahul-k-mishra/horizon.git"
WORKING = os.environ.get("HORIZON_WORKING", "/kaggle/working")
CODE = os.path.join(os.environ.get("HORIZON_TMP", "/tmp"), "horizon-code")
os.makedirs(WORKING, exist_ok=True)
LOG = open(os.path.join(WORKING, "console.log"), "w", encoding="utf-8")

def sh(*cmd):
    r = subprocess.run(list(cmd), capture_output=True, text=True)
    LOG.write(f"$ {' '.join(cmd)}\n{r.stdout}{r.stderr}")
    LOG.flush()
    if r.returncode:
        raise SystemExit(f"command failed with code {r.returncode}: {' '.join(cmd)}")
    return r.stdout

def freeze():
    return sorted(l for l in sh(sys.executable, "-m", "pip", "freeze", "--all").splitlines() if l.strip())

def norm(n):
    return re.sub(r"[-_.]+", "-", n).lower()

def tree(roots):
    """Every distribution the roots need, without extras, with its installed version."""
    seen, todo = {}, list(roots)
    while todo:
        n = norm(todo.pop())
        if n in seen:
            continue
        d = metadata.distribution(n)
        seen[n] = d.version
        for req in d.requires or []:
            if "extra ==" in req:
                continue
            spec, _, marker = req.partition(";")
            if marker.strip():
                env_ok = True
                try:
                    from packaging.markers import Marker
                    env_ok = Marker(marker.strip()).evaluate()
                except Exception:
                    pass
                if not env_ok:
                    continue
            todo.append(re.match(r"[A-Za-z0-9_.-]+", spec.strip()).group(0))
    return dict(sorted(seen.items()))

out = dict(docker_image=os.environ.get("KAGGLE_DOCKER_IMAGE"), python=sys.version, implementation=platform.python_implementation(),
           machine=platform.machine(), python_executable=sys.executable, code_tag=TAG)
out["freeze_before"] = freeze()
sh("git", "clone", "--quiet", REPO_URL, CODE)
sh("git", "-C", CODE, "checkout", "--quiet", TAG)
out["code_commit"] = sh("git", "-C", CODE, "rev-parse", "HEAD").strip()
REQ = os.path.join(CODE, "kaggle", "requirements-runner.txt")
out["requirements_runner"] = open(REQ, encoding="utf-8").read()
sh(sys.executable, "-m", "pip", "install", "--quiet", "-r", REQ)     # the runner's own command
out["freeze_after"] = freeze()
roots = [re.match(r"[A-Za-z0-9_.-]+", l).group(0) for l in out["requirements_runner"].splitlines()
         if l.strip() and not l.lstrip().startswith("#")]
out["runner_tree"] = tree(roots)
import gymnasium, minigrid                                                   # import only; no search
out["imported"] = dict(gymnasium=gymnasium.__version__, minigrid=getattr(minigrid, "__version__", None))
if os.path.exists(os.path.join(CODE, "scripts", "automation", "sim_gvgai.py")):
    t0 = time.time()
    import jpype
    out["imported"]["jpype"] = getattr(jpype, "__version__", None)
    out["image_java"] = dict(JAVA_HOME=os.environ.get("JAVA_HOME"), java=shutil.which("java"), javac=shutil.which("javac"))
    sys.path.insert(0, os.path.join(CODE, "scripts", "automation"))
    import sim_gvgai
    cache = os.path.join(os.environ.get("HORIZON_TMP", "/tmp"), "horizon-gvgai")
    jdk = sim_gvgai.ensure_jdk(cache)
    release = open(os.path.join(jdk, "release"), encoding="utf-8").read().splitlines()
    t_jdk = time.time() - t0
    bridge = sim_gvgai.start_jvm(cache)
    t_build = time.time() - t0 - t_jdk
    gf, lf = sim_gvgai.level_paths("corridor", 0, "scripts/automation/gvgai_testgames")
    root = bridge.load(gf, lf, 0)
    out["gvgai"] = dict(jdk_home=jdk, jdk_pinned=sim_gvgai.is_pinned_jdk(jdk),
                        jdk_release=[x for x in release if x.startswith(("IMPLEMENTOR", "JAVA_RUNTIME_VERSION"))],
                        jdk_archive_sha256=sim_gvgai.sha256_file(os.path.join(cache, sim_gvgai.TEMURIN["linux"]["name"])),
                        jdk_archive_pinned=sim_gvgai.TEMURIN["linux"]["sha256"],
                        java_sources=len(sim_gvgai.java_sources()),
                        corridor_0_actions=[str(a) for a in bridge.actionNames(root)],
                        corridor_0_key_bytes=len(bytes(bridge.key(root, False))),
                        seconds_jdk=round(t_jdk, 1), seconds_build_and_jvm=round(t_build, 1))
json.dump(out, open(os.path.join(WORKING, "probe.json"), "w", encoding="utf-8"), indent=1, sort_keys=True)
LOG.write("probe done\n")
LOG.close()
print("probe done")
